# Modular Task Configuration System

> **Experimental.** This notebook demonstrates the `TaskConfig` / `ModularTaskEnv` route,
> which is not maintained long-term. For new work, use the two supported backends instead:
> CPU `MyoGymnasiumEnv` + GPU mjlab `ManagerBasedRlEnvCfg` (see `docs/wiki/adding-a-new-task.md`).

The idea: define a task entirely through a `TaskConfig` dataclass — no subclassing of the
environment needed.

Topics covered:
1. Defining a task with `TaskConfig`
2. Running on the CPU backend (`ModularTaskEnv`)
3. Registering tasks with `register_task()` for `gymnasium.make()`
4. Callable obs/reward terms
5. Scene randomization
6. Action noise and muscle conditions (sarcopenia)


In [1]:
# Install dependencies if running in Colab
# !pip install myosuite

## 1. Imports

In [2]:
from dataclasses import dataclass, field

import numpy as np

from myosuite.core.config import (
    ActuatorGroupSpec,
    BackendConfig,
    GoalSpec,
    ObsSpec,
    RewardSpec,
    TaskConfig,
)
from myosuite.core.registry import register_task
from myosuite.envs.modular_env import ModularTaskEnv

## 2. Defining a Task

Subclass `TaskConfig` and override fields you care about.
All fields have sensible defaults — you only need to specify what differs.

In [3]:
@dataclass
class ElbowPoseTask(TaskConfig):
    """Reach a random elbow flexion angle."""

    model: str = "elbow_standard"
    obs: ObsSpec = field(
        default_factory=lambda: ObsSpec(keys=["joint_pos", "joint_vel"])
    )
    goal: GoalSpec = field(
        default_factory=lambda: GoalSpec(
            target_type="joint_angles",
            randomize=True,
            range={"r_elbow_flex": (0.0, 2.27)},
        )
    )
    reward: RewardSpec = field(default_factory=lambda: RewardSpec(terms=["pose"]))
    actuators: list[ActuatorGroupSpec] = field(
        default_factory=lambda: [ActuatorGroupSpec()]
    )
    max_episode_steps: int = 200
    backend: BackendConfig = field(
        default_factory=lambda: BackendConfig(n_substeps=5, ctrl_dt=0.005)
    )


task = ElbowPoseTask()
print(task)

ElbowPoseTask(model='elbow_standard', scene='flat_floor', max_episode_steps=200, muscle_fatigue=False, backend=BackendConfig(n_substeps=5, ctrl_dt=0.005, sim_dt=0.001, extra={}), obs=ObsSpec(keys=['joint_pos', 'joint_vel'], extra={}), goal=GoalSpec(target_type='joint_angles', randomize=True, range={'r_elbow_flex': (0.0, 2.27)}, extra={}), reward=RewardSpec(terms=['pose'], weights={}, extra={}), actuators=[ActuatorGroupSpec(name='muscles', actuator_type='muscle', condition='normal', normalize_actions=True, noise=0.0)])


## 3. Running on the CPU Backend

In [4]:
env = ModularTaskEnv(task)
print("Observation space:", env.observation_space)
print("Action space:     ", env.action_space)

obs, info = env.reset(seed=42)
print("\nObs shape:", obs.shape)
print("Goal (target_angles):", info.get("task_state", env._task_state).get("target_angles"))

Observation space: Box(-inf, inf, (2,), float32)
Action space:      Box(0.0, 1.0, (6,), float32)

Obs shape: (2,)
Goal (target_angles): [1.75688023]


njmax: parent has -1 (default), child has 1000, keeping parent value
nconmax: parent has -1 (default), child has 400, keeping parent value
nuser_jnt: parent has -1 (default), child has 1, keeping parent value

/Users/caggiano/Dropbox/Myo/myosuite4/myosuite/core/model_builder.py:844: UserWarning: Attach conflict when attaching 'MyoElbow_v0.1.7', policy is 'warning'
njmax: parent has -1 (default), child has 1000, keeping parent value
nconmax: parent has -1 (default), child has 400, keeping parent value
nuser_jnt: parent has -1 (default), child has 1, keeping parent value
  return spec.compile(), spec


In [5]:
# Run a short episode with random actions
total_reward = 0.0
for step in range(50):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

print(f"Steps run: {step + 1}")
print(f"Total reward: {total_reward:.4f}")
print(f"rwd_dict keys: {list(info['rwd_dict'].keys())}")

env.close()

Steps run: 50
Total reward: -71.2677
rwd_dict keys: ['pose/pose', 'pose/bonus', 'pose/penalty', 'pose/solved', 'dense', 'done', 'solved']


## 4. Registering with Gymnasium

`register_task()` wraps `gymnasium.register()` and returns the env id.
Registration is idempotent — safe to call multiple times.

In [6]:
import gymnasium as gym

# Auto-derive env id from class name
env_id = register_task(ElbowPoseTask())
print("Registered as:", env_id)

# Or specify explicitly
env_id = register_task(ElbowPoseTask(), env_id="MyElbowPose-v0")
print("Registered as:", env_id)

env = gym.make(env_id)
obs, _ = env.reset(seed=0)
print("obs.shape:", obs.shape)
env.close()

Registered as: ElbowPoseTask-v0
Registered as: MyElbowPose-v0
obs.shape: (2,)


njmax: parent has -1 (default), child has 1000, keeping parent value
nconmax: parent has -1 (default), child has 400, keeping parent value
nuser_jnt: parent has -1 (default), child has 1, keeping parent value



## 5. Callable Obs and Reward Terms

Instead of string keys, pass functions directly to `ObsSpec.keys` or `RewardSpec.terms`.
This is useful for custom terms that live outside the standard library.

In [7]:
from myosuite.terms.base_obs import joint_pos_obs, joint_vel_obs
from myosuite.terms.base_reward import pose_reward


@dataclass
class ElbowCallableTask(TaskConfig):
    """Same task but using callables instead of string keys."""

    model: str = "elbow_standard"
    obs: ObsSpec = field(
        # Functions accepted directly
        default_factory=lambda: ObsSpec(keys=[joint_pos_obs, joint_vel_obs])
    )
    goal: GoalSpec = field(
        default_factory=lambda: GoalSpec(
            target_type="joint_angles",
            randomize=True,
            range={"r_elbow_flex": (0.0, 2.27)},
        )
    )
    reward: RewardSpec = field(
        # Functions accepted directly
        default_factory=lambda: RewardSpec(terms=[pose_reward])
    )
    max_episode_steps: int = 50


env = ModularTaskEnv(ElbowCallableTask())
obs, _ = env.reset(seed=0)
_, reward, _, _, info = env.step(env.action_space.sample())
print(f"reward: {reward:.4f}")
env.close()

reward: -1.4367


njmax: parent has -1 (default), child has 1000, keeping parent value
nconmax: parent has -1 (default), child has 400, keeping parent value
nuser_jnt: parent has -1 (default), child has 1, keeping parent value



## 6. Scene Randomization

Set `scene` to a `list[str]` to sample a random scene at each episode reset.
All scene models are pre-compiled at init for fast episode resets.

In [8]:
@dataclass
class ElbowMultiSceneTask(ElbowPoseTask):
    """Randomize over two scene variants."""

    scene: list[str] = field(default_factory=lambda: ["flat_floor", "flat_floor"])


env = ModularTaskEnv(ElbowMultiSceneTask())

scenes_seen = set()
for i in range(10):
    obs, _ = env.reset(seed=i)
    if "scene" in env._task_state:
        scenes_seen.add(env._task_state["scene"])

print("Scenes encountered:", scenes_seen)
env.close()

Scenes encountered: {np.str_('flat_floor')}


njmax: parent has -1 (default), child has 1000, keeping parent value
nconmax: parent has -1 (default), child has 400, keeping parent value
nuser_jnt: parent has -1 (default), child has 1, keeping parent value

njmax: parent has -1 (default), child has 1000, keeping parent value
nconmax: parent has -1 (default), child has 400, keeping parent value
nuser_jnt: parent has -1 (default), child has 1, keeping parent value



## 7. Action Noise and Muscle Conditions

`ActuatorGroupSpec.noise` adds Gaussian noise to actions at each step — useful for domain randomization.
`ActuatorGroupSpec.condition = "sarcopenia"` applies muscle strength reduction at model build time.

In [9]:
# Action noise
@dataclass
class ElbowNoisyTask(ElbowPoseTask):
    """Elbow task with Gaussian action noise for domain randomization."""

    actuators: list[ActuatorGroupSpec] = field(
        default_factory=lambda: [ActuatorGroupSpec(noise=0.05)]
    )


env = ModularTaskEnv(ElbowNoisyTask())
env.reset(seed=0)
zero_action = np.zeros(env.action_space.shape, dtype=np.float32)
_, _, _, _, info = env.step(zero_action)
print("rwd_dict:", {k: round(float(v), 6) for k, v in info["rwd_dict"].items() if isinstance(v, (int, float))})
env.close()

rwd_dict: {'pose/pose': -1.444472, 'pose/bonus': 0.0, 'pose/penalty': -0.0, 'dense': -1.444472, 'done': 0.0, 'solved': 0.0}


njmax: parent has -1 (default), child has 1000, keeping parent value
nconmax: parent has -1 (default), child has 400, keeping parent value
nuser_jnt: parent has -1 (default), child has 1, keeping parent value



In [10]:
# Sarcopenia (reduced muscle strength)
@dataclass
class ElbowSarcopeniaTask(ElbowPoseTask):
    """Elbow task with simulated age-related muscle weakness."""

    actuators: list[ActuatorGroupSpec] = field(
        default_factory=lambda: [ActuatorGroupSpec(condition="sarcopenia")]
    )


env = ModularTaskEnv(ElbowSarcopeniaTask())
obs, _ = env.reset(seed=0)
print("Sarcopenia env obs shape:", obs.shape)
print("Action space:", env.action_space)
env.close()

Sarcopenia env obs shape: (2,)
Action space: Box(0.0, 1.0, (6,), float32)


njmax: parent has -1 (default), child has 1000, keeping parent value
nconmax: parent has -1 (default), child has 400, keeping parent value
nuser_jnt: parent has -1 (default), child has 1, keeping parent value

/Users/caggiano/Dropbox/Myo/myosuite4/myosuite/envs/modular_env.py:295: UserWarning: Attach conflict when attaching 'MyoElbow_v0.1.7', policy is 'warning'
njmax: parent has -1 (default), child has 1000, keeping parent value
nconmax: parent has -1 (default), child has 400, keeping parent value
nuser_jnt: parent has -1 (default), child has 1, keeping parent value
  self.model = base_spec.compile()


## 8. Task Variants via Subclassing

The real power of `TaskConfig` is defining task families through single-field overrides.
No environment code changes — only data.

In [11]:
@dataclass
class ElbowPoseHardTask(ElbowPoseTask):
    """Harder variant: shorter episodes, full ROM range."""

    max_episode_steps: int = 50
    goal: GoalSpec = field(
        default_factory=lambda: GoalSpec(
            target_type="joint_angles",
            randomize=True,
            range={"r_elbow_flex": (0.0, 2.80)},  # extended range
        )
    )
    reward: RewardSpec = field(
        default_factory=lambda: RewardSpec(
            terms=["pose", "act_reg"],
            weights={"pose": 2.0, "act_reg": 0.1},  # stronger pose pressure
        )
    )


# Register all variants with one call each
easy_id = register_task(ElbowPoseTask(), env_id="ElbowPoseEasy-v0")
hard_id = register_task(ElbowPoseHardTask(), env_id="ElbowPoseHard-v0")
sarc_id = register_task(ElbowSarcopeniaTask(), env_id="ElbowPoseSarcopenia-v0")

print("Registered environments:")
for eid in [easy_id, hard_id, sarc_id]:
    env = gym.make(eid)
    obs, _ = env.reset(seed=0)
    print(f"  {eid}: obs_dim={obs.shape[0]}, n_act={env.action_space.shape[0]}")
    env.close()

Registered environments:


njmax: parent has -1 (default), child has 1000, keeping parent value
nconmax: parent has -1 (default), child has 400, keeping parent value
nuser_jnt: parent has -1 (default), child has 1, keeping parent value



  ElbowPoseEasy-v0: obs_dim=2, n_act=6
  ElbowPoseHard-v0: obs_dim=2, n_act=6


njmax: parent has -1 (default), child has 1000, keeping parent value
nconmax: parent has -1 (default), child has 400, keeping parent value
nuser_jnt: parent has -1 (default), child has 1, keeping parent value

njmax: parent has -1 (default), child has 1000, keeping parent value
nconmax: parent has -1 (default), child has 400, keeping parent value
nuser_jnt: parent has -1 (default), child has 1, keeping parent value



  ElbowPoseSarcopenia-v0: obs_dim=2, n_act=6


## Summary

The Modular Task Configuration System lets you:

| Feature | API |
|---|---|
| Define a task | `@dataclass class MyTask(TaskConfig)` |
| Custom obs | `ObsSpec(keys=["joint_pos", my_fn])` |
| Custom reward | `RewardSpec(terms=["pose", my_fn], weights={...})` |
| Goal sampling | `GoalSpec(target_type="joint_angles", range={...})` |
| Scene randomization | `scene=["scene_a", "scene_b"]` |
| Sarcopenia / fatigue | `ActuatorGroupSpec(condition="sarcopenia")` |
| Action noise | `ActuatorGroupSpec(noise=0.05)` |
| Gymnasium registration | `register_task(MyTask(), env_id="MyTask-v0")` |

All three backends share the same `TaskConfig`:
- **CPU**: `ModularTaskEnv(task_config)`
- **MJX**: `MjxModularTaskEnv(task_config)` (requires JAX)
- **mjlab**: `make_modular_mjlab_cfg(task_config)` → pass to mjlab's `ManagerBasedRlEnv`